# Pollution Fingerprint Analysis

Phân tích kiểu ô nhiễm đặc trưng của từng thành phố bằng city-level pollution profile, chuẩn hóa, K-Means clustering và cosine similarity.

## 1. Mục tiêu

- Xây dựng pollution profile cho từng city.
- Chuẩn hóa 6 pollutant: PM2.5, PM10, NO2, SO2, O3, CO.
- Tìm số cluster phù hợp bằng Silhouette Score.
- Phân nhóm các city theo fingerprint.
- Tìm các city có profile tương đồng bằng cosine similarity.
- Khám phá sự phân bố địa lý của các cluster.

In [ ]:
import pandas as pd\nimport numpy as np\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.cluster import KMeans\nfrom sklearn.metrics import silhouette_score\nfrom sklearn.metrics.pairwise import cosine_similarity\n\ndf = pd.read_csv('../data/raw/globalAirQuality.csv')\npollutants = ['pm25', 'pm10', 'no2', 'so2', 'o3', 'co']\ndf.head()

## 2. City Pollution Profile

In [ ]:
city_profile = (\n    df.groupby(['city', 'country'], as_index=False)[pollutants]\n      .mean()\n)\ncity_profile

## 3. Standardization

In [ ]:
scaler = StandardScaler()\nX = scaler.fit_transform(city_profile[pollutants])

## 4. Chọn số cluster bằng Silhouette Score

In [ ]:
scores = {}\nfor k in range(2, 9):\n    km = KMeans(n_clusters=k, random_state=42, n_init=20)\n    labels = km.fit_predict(X)\n    scores[k] = silhouette_score(X, labels)\n\npd.Series(scores, name='silhouette_score')

## 5. K-Means và Pollution Fingerprint

In [ ]:
best_k = max(scores, key=scores.get)\nkmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)\nlabels = kmeans.fit_predict(X)\n\nfingerprint = city_profile.copy()\nfor i, col in enumerate(pollutants):\n    fingerprint[f'{col}_z'] = X[:, i]\nfingerprint['cluster'] = labels\nfingerprint

## 6. Diễn giải cluster

In [ ]:
cluster_means = fingerprint.groupby('cluster')[[f'{p}_z' for p in pollutants]].mean()\ncluster_means

## 7. City Similarity

In [ ]:
similarity = cosine_similarity(X)\nsimilarity_df = pd.DataFrame(\n    similarity,\n    index=city_profile['city'],\n    columns=city_profile['city']\n)\nsimilarity_df.head()

## 8. Key Findings

Sau khi chạy notebook, tập trung trả lời 3 câu hỏi:\n\n1. Có bao nhiêu pollution profile khác nhau?\n2. Pollutant nào phân biệt các cluster rõ nhất?\n3. Thành phố nào có fingerprint tương đồng nhất?